<a href="https://colab.research.google.com/github/Kashaf537/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kashaf537/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## Ranked actions + reason codes

The model output is treated as a decision-support queue rather than an automatic content-update system.

Pages are ranked using the Random Forest score for `is_declining_label`. Higher scores indicate that the model considers the page more likely to be declining.

Each recommendation is paired with simple reason codes based on observable content and performance signals. These reason codes are intended to help a human reviewer understand why a page was prioritized.

### Action priority

1. **Refresh content** — high-confidence declining pages with evidence of weak or worsening performance.
2. **Investigate performance** — pages with declining signals but weaker model confidence or conflicting indicators.
3. **Monitor** — pages showing some risk but insufficient evidence for immediate action.
4. **No immediate action** — pages with low decline risk.

### Example reason codes

- `HIGH_DECLINE_SCORE` — model assigns a high probability of decline.
- `LOW_CTR` — click-through rate is relatively low.
- `POOR_POSITION` — average search position is weak.
- `LOW_ENGAGEMENT` — engagement rate is low.
- `LOW_SCROLL` — scroll rate is low.
- `STALE_CONTENT` — content has not been updated recently.
- `TRAFFIC_DECLINE` — recent traffic/impression signals indicate deterioration.

The reason codes are supporting evidence, not proof that a particular change will improve performance.

In [1]:
# Clone the repository
!git clone https://github.com/Kashaf537/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 221, done.
remote: Counting objects: 100% (221/221), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 221 (delta 97), reused 123 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (221/221), 2.64 MiB | 14.87 MiB/s, done.
Resolving deltas: 100% (97/97), done.


In [2]:
# Move into the repository
%cd /content/flyrank-ml-internship

/content/flyrank-ml-internship


In [3]:
!ls

AGENTS.md  DATA_USE.md	LICENSE    paper	     scripts   submission
CLAUDE.md  docs		notebooks  README.md	     SETUP.md  work
data	   GUIDE.md	outputs    requirements.txt  skills    workflows


In [9]:
from pathlib import Path
import pandas as pd

df = Path("data/raw/content_refresh_anonymized.csv")

print("Dataset exists:", df.exists())

if df.exists():
    df = pd.read_csv(df)

    print("Shape:", df.shape)
    print("Columns:", len(df.columns))
    print("\nFirst 5 rows:")
    display(df.head())
    print(df.columns)
else:
    print("Dataset not found!")

Dataset exists: True
Shape: (30000, 44)
Columns: 44

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')


In [10]:
import pandas as pd
import numpy as np

print("Dataset shape:", df.shape)
print("Number of columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (30000, 44)
Number of columns: 44

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [11]:
# Create declining label from trend percentage
# Declining = more than 10% decrease

df["is_declining_label"] = (
    df["trend_pct"] < -10
).astype(int)

print("Target created successfully.")

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget percentages:")
print(df["is_declining_label"].value_counts(normalize=True) * 100)

Target created successfully.

Target distribution:
is_declining_label
1    18248
0    11752
Name: count, dtype: int64

Target percentages:
is_declining_label
1    60.826667
0    39.173333
Name: proportion, dtype: float64


In [12]:
target = "is_declining_label"

excluded_columns = [
    "content_id",
    "client_id",
    "trend_pct",
    "trend_direction",
    target
]

feature_columns = [
    col for col in df.columns
    if col not in excluded_columns
]

print("Number of model features:", len(feature_columns))
print("\nFeatures:")
print(feature_columns)

Number of model features: 40

Features:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']


In [13]:
model_df = df.dropna(subset=[target]).copy()

X = model_df[feature_columns].copy()
y = model_df[target].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (30000, 40)
y shape: (30000,)

Target distribution:
is_declining_label
1    18248
0    11752
Name: count, dtype: int64


In [14]:
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

Numeric features: 29
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Categorical features: 11
['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']


In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training rows: 24000
Testing rows: 6000

Training target distribution:
is_declining_label
1    14598
0     9402
Name: count, dtype: int64

Testing target distribution:
is_declining_label
1    3650
0    2350
Name: count, dtype: int64


In [17]:
from sklearn.ensemble import RandomForestClassifier

rf_classifier = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", rf_classifier)
])

print("Random Forest pipeline created.")

Random Forest pipeline created.


In [18]:
rf_model.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    classification_report
)

y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print("Random Forest evaluation")
print("------------------------")
print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

Random Forest evaluation
------------------------
Accuracy : 0.867
Precision: 0.844
Recall   : 0.958

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.73      0.81      2350
           1       0.84      0.96      0.90      3650

    accuracy                           0.87      6000
   macro avg       0.88      0.84      0.85      6000
weighted avg       0.87      0.87      0.86      6000



In [20]:
# Probability of being a declining page
model_df["rf_score"] = rf_model.predict_proba(X)[:, 1]

# Rank pages from highest to lowest decline score
model_df = model_df.sort_values(
    "rf_score",
    ascending=False
).reset_index(drop=True)

model_df["rf_rank"] = np.arange(
    1,
    len(model_df) + 1
)

print("Scores generated successfully.")

display(
    model_df[
        [
            "content_id",
            "client_id",
            "rf_score",
            "rf_rank",
            "is_declining_label"
        ]
    ].head(20)
)

Scores generated successfully.


,content_id,client_id,rf_score,rf_rank,is_declining_label
0,content_28494292dac5,client_7f2253d7e2,1.0,1,1
1,content_30063a4bbeeb,client_7f2253d7e2,1.0,2,1
2,content_0868e82484db,client_7f2253d7e2,1.0,3,1
3,content_9f180a1406cc,client_8722616204,1.0,4,1
4,content_483e3d6fede3,client_7f2253d7e2,1.0,5,1
5,content_76ab1e37b829,client_3fdba35f04,1.0,6,1
6,content_4d1295d31a4b,client_3fdba35f04,1.0,7,1
7,content_8b9c2a589d2c,client_7f2253d7e2,1.0,8,1
8,content_d1e879090911,client_a88a7902cb,1.0,9,1
9,content_8dba22b835f8,client_3fdba35f04,1.0,10,1


In [21]:
def make_reason_codes(row):

    reasons = []

    # Model confidence
    if row["rf_score"] >= 0.75:
        reasons.append("HIGH_DECLINE_SCORE")

    elif row["rf_score"] >= 0.50:
        reasons.append("MODERATE_DECLINE_SCORE")

    # CTR
    if pd.notna(row["ctr"]) and row["ctr"] < 0.10:
        reasons.append("LOW_CTR")

    # Search position
    if pd.notna(row["avg_position"]) and row["avg_position"] > 20:
        reasons.append("POOR_POSITION")

    # Engagement
    if (
        pd.notna(row["engagement_rate"])
        and row["engagement_rate"] < 1
    ):
        reasons.append("LOW_ENGAGEMENT")

    # Scroll
    if (
        pd.notna(row["scroll_rate"])
        and row["scroll_rate"] < 10
    ):
        reasons.append("LOW_SCROLL")

    # Content freshness
    if (
        pd.notna(row["days_since_last_update"])
        and row["days_since_last_update"] > 180
    ):
        reasons.append("STALE_CONTENT")

    # Recent impressions lower than previous period
    if (
        pd.notna(row["impressions_last_30d"])
        and pd.notna(row["impressions_prev_30d"])
        and row["impressions_last_30d"]
        < row["impressions_prev_30d"]
    ):
        reasons.append("RECENT_IMPRESSION_DECLINE")

    if len(reasons) == 0:
        reasons.append("REVIEW_REQUIRED")

    return ", ".join(reasons)


model_df["reason_codes"] = model_df.apply(
    make_reason_codes,
    axis=1
)

print("Reason codes created.")

display(
    model_df[
        [
            "content_id",
            "rf_score",
            "reason_codes"
        ]
    ].head(20)
)

Reason codes created.


,content_id,rf_score,reason_codes
0,content_28494292dac5,1.0,"HIGH_DECLINE_SCORE, POOR_POSITION, LOW_ENGAGEM..."
1,content_30063a4bbeeb,1.0,"HIGH_DECLINE_SCORE, LOW_CTR, LOW_ENGAGEMENT, R..."
2,content_0868e82484db,1.0,"HIGH_DECLINE_SCORE, LOW_CTR, LOW_ENGAGEMENT, L..."
3,content_9f180a1406cc,1.0,"HIGH_DECLINE_SCORE, LOW_CTR, LOW_ENGAGEMENT, L..."
4,content_483e3d6fede3,1.0,"HIGH_DECLINE_SCORE, LOW_CTR, LOW_ENGAGEMENT, L..."
5,content_76ab1e37b829,1.0,"HIGH_DECLINE_SCORE, LOW_CTR, LOW_ENGAGEMENT, R..."
6,content_4d1295d31a4b,1.0,"HIGH_DECLINE_SCORE, LOW_CTR, POOR_POSITION, LO..."
7,content_8b9c2a589d2c,1.0,"HIGH_DECLINE_SCORE, LOW_CTR, RECENT_IMPRESSION..."
8,content_d1e879090911,1.0,"HIGH_DECLINE_SCORE, LOW_CTR, POOR_POSITION, LO..."
9,content_8dba22b835f8,1.0,"HIGH_DECLINE_SCORE, LOW_CTR, POOR_POSITION, LO..."


In [22]:
def assign_action(score):

    if score >= 0.75:
        return "REFRESH_CONTENT"

    elif score >= 0.50:
        return "INVESTIGATE"

    elif score >= 0.25:
        return "MONITOR"

    else:
        return "NO_IMMEDIATE_ACTION"


model_df["recommended_action"] = (
    model_df["rf_score"]
    .apply(assign_action)
)

print("Actions assigned.")

print(
    model_df["recommended_action"]
    .value_counts()
)

Actions assigned.
recommended_action
REFRESH_CONTENT        17011
NO_IMMEDIATE_ACTION     9595
INVESTIGATE             1740
MONITOR                 1654
Name: count, dtype: int64


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## Intended use and limits

### Intended use

This analysis is intended as a **decision-support tool for SEO/content teams**.

The ranked queue can help a content strategist or SEO specialist:

- Prioritize pages that may need attention.
- Identify pages showing signals associated with declining performance.
- Review content age, recent impressions, clicks, search demand, and engagement signals.
- Decide which pages should be investigated for a possible refresh.
- Support human prioritization when the number of pages is large.

The model output should be treated as a **prioritized review queue**, not as an automatic decision about what a page must do.

### What the model measures

The model uses observable content and performance signals such as:

- Search volume
- Competition
- Content age
- Days with impressions
- Impressions and clicks
- Sessions and engaged sessions
- Recent versus previous-period performance
- CTR
- Average position
- Engagement rate
- Scroll rate
- AI traffic percentage

These features describe patterns in the available dataset. They do not prove that changing a particular feature will cause rankings or traffic to improve.

### Limits

The model should not be interpreted as a causal SEO model.

A high model score means that a page is ranked highly by the model for review. It does **not** mean that the page will definitely decline or that a specific intervention will definitely improve performance.

The analysis is also limited by the available dataset and its historical observations. Results may not generalize to:

- New clients not represented in the data.
- Websites with substantially different content types.
- Major changes in Google's ranking systems.
- Pages with very little historical traffic data.
- Situations where external factors explain the performance change.

The model should therefore be used as **directional decision-support**, with final actions determined by a human reviewer.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review rules

Every page in the ranked queue should be reviewed by an SEO/content specialist before an action is taken.

The reviewer should check:

1. **Search intent**
   - Does the page still match the user's search intent?
   - Has the search intent changed?

2. **Content freshness**
   - Is the information outdated?
   - Have important facts, products, services, or statistics changed?

3. **Search performance**
   - Are impressions or clicks actually declining?
   - Has average position changed?
   - Is the observed change large enough to matter?

4. **Content quality**
   - Is the page useful and complete?
   - Are there missing topics or questions?
   - Is the content substantially weaker than competing results?

5. **Business context**
   - Is the page strategically important?
   - Does it support an important product, service, or business objective?

6. **Possible external causes**
   - Could seasonality, algorithm changes, technical problems, or tracking changes explain the observed pattern?

### Recommended action categories

After review, a page can be assigned to one of the following actions:

- **Refresh** — update outdated or incomplete information.
- **Improve** — strengthen content depth, structure, or search-intent alignment.
- **Monitor** — performance shows a signal but there is not enough evidence for immediate action.
- **Investigate** — possible technical, ranking, tracking, or external issue.
- **No action** — the model signal is not supported by human review.

### No-go list

The following decisions should **not be automated** using this model:

- Automatically deleting content.
- Automatically rewriting or publishing content.
- Automatically changing search intent.
- Automatically changing prices or business-critical information.
- Automatically redirecting or canonicalizing pages.
- Automatically deciding that a page is low quality.
- Automatically declaring an SEO strategy successful or unsuccessful.
- Automatically making high-impact technical SEO changes.
- Automatically treating model predictions as proof of future traffic or ranking changes.

The model should only prioritize pages for human investigation and support a structured review process.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The recommendation system should be monitored because SEO behavior and search environments can change over time.

### Monitoring signals

The following should be monitored periodically:

- Precision@50 of the ranked queue when labels become available.
- Number of genuinely declining pages appearing in the top-ranked results.
- False-positive rate in the review queue.
- Distribution of model scores.
- Distribution of important input features.
- Changes in the proportion of declining versus non-declining pages.
- Changes in traffic and impression patterns.
- Changes in client or content-type composition.

### Potential model-staleness signals

The model should be investigated if:

- Precision@50 decreases substantially compared with the current measured result.
- The top-ranked queue contains increasingly many pages that human reviewers consider healthy.
- False positives increase consistently.
- The distribution of important features changes substantially.
- The proportion of declining pages changes significantly.
- New clients or content types behave differently from the data used to train the model.
- Search behavior or external SEO conditions change substantially.

### Retraining triggers

Retraining should be considered when:

1. A sufficiently large amount of new labeled data becomes available.
2. Model performance deteriorates on recent observations.
3. Feature distributions change substantially.
4. The content portfolio changes significantly.
5. New clients or content types are introduced.
6. Search-engine behavior changes enough that historical relationships may no longer represent current conditions.

Retraining should be performed using a **time-aware or client-grouped validation strategy** where possible, rather than relying only on a random split.

### Monitoring principle

The system should be treated as a periodically reviewed decision-support model rather than a permanent production rule.

Observed performance should be measured on newer data before making claims that the model continues to generalize.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [24]:
action_queue = model_df[
    [
        "rf_rank",
        "content_id",
        "client_id",
        "rf_score",
        "recommended_action",
        "reason_codes"
    ]
].head(50).copy()

print("Top 50 ranked action queue:")
print("Shape:", action_queue.shape)

display(action_queue)

Top 50 ranked action queue:
Shape: (50, 6)


,rf_rank,content_id,client_id,rf_score,recommended_action,reason_codes
0,1,content_28494292dac5,client_7f2253d7e2,1.0,REFRESH_CONTENT,"HIGH_DECLINE_SCORE, POOR_POSITION, LOW_ENGAGEM..."
1,2,content_30063a4bbeeb,client_7f2253d7e2,1.0,REFRESH_CONTENT,"HIGH_DECLINE_SCORE, LOW_CTR, LOW_ENGAGEMENT, R..."
2,3,content_0868e82484db,client_7f2253d7e2,1.0,REFRESH_CONTENT,"HIGH_DECLINE_SCORE, LOW_CTR, LOW_ENGAGEMENT, L..."
3,4,content_9f180a1406cc,client_8722616204,1.0,REFRESH_CONTENT,"HIGH_DECLINE_SCORE, LOW_CTR, LOW_ENGAGEMENT, L..."
4,5,content_483e3d6fede3,client_7f2253d7e2,1.0,REFRESH_CONTENT,"HIGH_DECLINE_SCORE, LOW_CTR, LOW_ENGAGEMENT, L..."
5,6,content_76ab1e37b829,client_3fdba35f04,1.0,REFRESH_CONTENT,"HIGH_DECLINE_SCORE, LOW_CTR, LOW_ENGAGEMENT, R..."
6,7,content_4d1295d31a4b,client_3fdba35f04,1.0,REFRESH_CONTENT,"HIGH_DECLINE_SCORE, LOW_CTR, POOR_POSITION, LO..."
7,8,content_8b9c2a589d2c,client_7f2253d7e2,1.0,REFRESH_CONTENT,"HIGH_DECLINE_SCORE, LOW_CTR, RECENT_IMPRESSION..."
8,9,content_d1e879090911,client_a88a7902cb,1.0,REFRESH_CONTENT,"HIGH_DECLINE_SCORE, LOW_CTR, POOR_POSITION, LO..."
9,10,content_8dba22b835f8,client_3fdba35f04,1.0,REFRESH_CONTENT,"HIGH_DECLINE_SCORE, LOW_CTR, POOR_POSITION, LO..."


In [25]:
from pathlib import Path

# Create output directory if it does not exist
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Export the ranked action queue
output_path = output_dir / "content_action_queue.csv"
action_queue.to_csv(output_path, index=False)

print(f"Queue exported successfully to: {output_path}")
print("Exported rows:", len(action_queue))

Queue exported successfully to: work/outputs/content_action_queue.csv
Exported rows: 50


In [26]:
print("File exists:", output_path.exists())
print("File size:", output_path.stat().st_size, "bytes")

File exists: True
File size: 6908 bytes


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.